# Build 2: hotel overbooking — beginner walkthrough

This notebook walks through the uploaded CSV and calls the reproducible Python script. **The capacities are hypothetical; results are a retrospective scenario, not real-world recommendations.**

## Step 1 — Read the file
`pd.read_csv` reads each row as a reservation outcome. `head` previews a few records.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
df = pd.read_csv("../hotels.csv")
print(df.shape)
display(df.head())

## Step 2 — Understand data quality
`isna().sum()` counts missing values; `duplicated()` counts *exact matches*, NOT confirmed duplicate bookings.

In [ ]:
display(df.isna().sum().sort_values(ascending=False).head(12))
print("Matching rows:", df.duplicated().sum())
print(df.groupby("hotel")["is_canceled"].agg(["size", "mean"]))

## Step 3 — Create arrival date
Combine year, English month name and day to a datetime. This enables chronological splitting.

In [ ]:
df["arrival_date"] = pd.to_datetime(df["arrival_date_year"].astype(str)+"-"+df["arrival_date_month"]+"-"+df["arrival_date_day_of_month"].astype(str),format="%Y-%B-%d")
df["shows"] = 1 - df["is_canceled"]
print(df["arrival_date"].min(), df["arrival_date"].max())

## Step 4 — Split by time
The 2017 test is kept unseen during selection.

In [ ]:
train = df[df.arrival_date < "2016-07-01"]
validation = df[(df.arrival_date >= "2016-07-01") & (df.arrival_date < "2017-01-01")]
test = df[df.arrival_date >= "2017-01-01"]
print(len(train),len(validation),len(test))

## Step 5 — Understand the money
For each eligible night, `capacity` is an **assumed** scenario value, `shows` is observed arrival count, and `k` is extra guaranteed-show guests in this simplified simulator. This is not realistic multiday occupancy.

In [ ]:
capacity, shows, k, night_value = 20, 18, 4, 100
empty = max(0,capacity-shows-k)
walked = max(0,shows+k-capacity)
loss = empty*night_value + walked*300
print({"empty":empty,"walked":walked,"loss_eur":loss})

## Step 6 — Train safely
See `build2_analysis.py`: only booking-plausible inputs enter a preprocessing pipeline and a LogisticRegression model. Final status and status date are explicitly excluded.

In [ ]:
from build2_analysis import NUM, CAT, LEAKY
print("Numeric features", NUM)
print("Categorical features", CAT)
print("Excluded potentially leaky features", LEAKY)

## Step 7 — Run the full experiment
The command below freezes each rule using H2 2016 and writes the 2017 score table once. It may take a few seconds depending on machine and installed packages.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"build2_analysis.py","--csv","../hotels.csv","--out","results","--city-capacity","150","--resort-capacity","100"],check=True)

## Step 8 — Compare frozen policies

In [ ]:
results = pd.read_csv("results/test_comparison.csv")
display(results[["hotel","method","parameter_frozen","cost_eur","walks_per_1000","savings_vs_zero_eur","test_under_ceiling"]])

## Step 9 — Explain the caveat to your panel
We do **not** have real capacities, booking-state cutoffs, cancellation timestamps, or a unique booking ID. We also use arrival-night cohorts rather than all nights of stay. All euro numbers are scenario outcomes only. Do not optimize the final rule on 2017 after seeing test metrics.

## Step 10 — Try a sensitivity run
Change only assumed capacities below; compare outcomes as distinct **scenarios**, not a test-set optimization. For rigorous estimates, obtain the hotel's actual daily capacity and snapshots.

In [ ]:
# Example scenario (run only when you want to examine assumption sensitivity):
# subprocess.run([sys.executable,"build2_analysis.py","--csv","../hotels.csv",
#                 "--out","results_capacity_scenario","--city-capacity","170",
#                 "--resort-capacity","110"],check=True)